## Libraries import

In [1]:
import os
from PIL import Image
from torch.utils.data import Dataset, Subset, DataLoader
from torchvision import transforms
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import accuracy_score
from tqdm import tqdm
import copy

## Dataset Class

In [2]:
class CelebADataset(Dataset):
    def __init__(self, label_file, image_dir, partition_file, transform=None):
        self.image_dir = image_dir
        self.transform = transform
        self.samples = []
        self.partition_map = {}
        self.train_indices = []
        self.val_indices = []
        self.test_indices = []

        with open(partition_file, 'r') as f:
            for line in f:
                img_name, partition = line.strip().split()
                base_name = os.path.splitext(img_name)[0]
                self.partition_map[base_name] = int(partition)

        with open(label_file, 'r') as f:
            for line in f:
                img_name, label = line.strip().split()
                self.samples.append((img_name, int(label)))

        for idx, (img_name, _) in enumerate(self.samples):
            base_name = os.path.splitext(img_name)[0]
            split = self.partition_map[base_name]
            if split == 0:
                self.train_indices.append(idx)
            elif split == 1:
                self.val_indices.append(idx)
            elif split == 2:
                self.test_indices.append(idx)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_name, label = self.samples[idx]
        img_path = os.path.join(self.image_dir, img_name)
        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image, label

## Data Loading & Transform

In [3]:
base_path = '../AdvCelebA'
label_file = os.path.join(base_path, 'attack_CelebA.txt')
image_dir = os.path.join(base_path, 'images')
partition_file = os.path.join(base_path, 'list_eval_partition_no_overlap.txt')

transform = transforms.ToTensor()

dataset = CelebADataset(label_file, image_dir, partition_file, transform=transform)

train_dataset = Subset(dataset, dataset.train_indices)
val_dataset = Subset(dataset, dataset.val_indices)
test_dataset = Subset(dataset, dataset.test_indices)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32)
test_loader = DataLoader(val_dataset, batch_size=32)

## Model (Simple CNN)

In [8]:
class BinaryClassifier(nn.Module):
    def __init__(self):
        super(BinaryClassifier, self).__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1), nn.BatchNorm2d(16), nn.ReLU(), nn.MaxPool2d(2),   # -> (16, 56, 56)
            nn.Conv2d(16, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),  # -> (32, 28, 28)
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),  # -> (64, 14, 14)
        )
        self.fc = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),  
            nn.Flatten(),                 
            nn.Linear(64, 128),          
            nn.Dropout(0.5),
            nn.Linear(128, 1)         
        )

    def forward(self, x):
        x = self.conv(x)
        x = self.fc(x)
        return x

## Training Loop

In [9]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = BinaryClassifier().to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

epochs = 30
patience = 5
best_val_loss = float('inf')
early_stop_counter = 0
best_model_wts = copy.deepcopy(model.state_dict())

for epoch in range(epochs):
    model.train()
    train_loss = 0.0
    for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1} - Training"):
        images = images.to(device)
        labels = labels.float().unsqueeze(1).to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item() * images.size(0)

    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.float().unsqueeze(1).to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * images.size(0)

    train_loss /= len(train_loader.dataset)
    val_loss /= len(val_loader.dataset)
    print(f"Epoch {epoch+1}, Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}")

    # Early stopping check
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_model_wts = copy.deepcopy(model.state_dict())
        early_stop_counter = 0
    else:
        early_stop_counter += 1
        print(f"🔁 No improvement in validation loss for {early_stop_counter} epochs.")

        if early_stop_counter >= patience:
            print(f"⏹️ Early stopping triggered at epoch {epoch+1}")
            break

# Load best model weights
model.load_state_dict(best_model_wts)

Epoch 1 - Training: 100%|██████████| 5072/5072 [08:54<00:00,  9.48it/s]


Epoch 1, Train Loss: 0.3515, Val Loss: 0.3146


Epoch 2 - Training: 100%|██████████| 5072/5072 [09:23<00:00,  9.01it/s]


Epoch 2, Train Loss: 0.3121, Val Loss: 0.3143


Epoch 3 - Training: 100%|██████████| 5072/5072 [09:49<00:00,  8.60it/s]


...

Epoch 4, Train Loss: 0.2927, Val Loss: 0.2812


Epoch 5 - Training: 100%|██████████| 5072/5072 [10:02<00:00,  8.42it/s]


Epoch 5, Train Loss: 0.2828, Val Loss: 0.2876
🔁 No improvement in validation loss for 1 epochs.


Epoch 6 - Training:  48%|████▊     | 2434/5072 [04:49<05:13,  8.41it/s]


KeyboardInterrupt: 

In [10]:
model.load_state_dict(best_model_wts)

<All keys matched successfully>

In [11]:
torch.save(model.state_dict(), '../models_bin/cnn.pth')

## Validation Accuracy

In [12]:
model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)
        labels = labels.to(device)
        outputs = model(images)
        preds = torch.sigmoid(outputs).cpu().numpy() > 0.5
        all_preds.extend(preds.flatten())
        all_labels.extend(labels.cpu().numpy())

acc = accuracy_score(all_labels, all_preds)
print(f"Validation Accuracy: {acc:.4f}")

Validation Accuracy: 0.8873
